# Checking Candidate Codebooks

---

Use this notebook to check candidate codebook pairs (maybe eventually more users?) for a real-addition binary input adversarial MAC. We are checking whether we can accomplish partial correction. 
- With two users, this will always mean correcting at least one of the two users when the adversary acts, and both when the adversary does not act.
- With more users, we will need to add a threshold we wish to correct.

In [1]:
import numpy as np
import math
from itertools import product

Good Codebook Pair Checker
---

Some helper functions before we define the big function:

In [2]:
def unique(list1):
    #imput a list
    # return a list containing the unique values of the input
    list_set = set(list1)
    unique_list = list(list_set)
    return unique_list


def generate_words(n, adv=False):
    #Generates all possible codewords and adversary words of length n
    #Excludes all 0 from both, all 1's from codewords due to support constraints
    
    words = [int(np.binary_repr(z)) for z in range(2**n)]
    
    if not adv:
        words.pop(2**n-1)
        
    words.pop(0)

    return words


#takes a list of binary strings (or ints or floats) and converts entries to integers
integer_list = lambda list_binary : [int(ii) for ii in list_binary]




def zero_pad(n, integer): 
    #pads integers with zeros so they look like the same block length
    #return is a string 
    if (len(str(integer)) == n):
        return str(integer) 
    else: 
        zeros = n-len(str(integer))
        return zeros*'0'+str(integer)

#zero pad all the integers in an iterable (list, set, tuple, etc.). Return is a list.
zero_pad_iter = lambda n, my_iter: [zero_pad(n, integer) for integer in my_iter]

Now, the checker function

In [3]:
#given a codebook pair and adversarial options, check whether the codebook is good for partial correction
def good_pair_check(cb1, cb2, n):
    #cb1, cb2 are lists of binary words of binary length n
    #binary codewords in list can be input as strings to allow leading 0's
    
    #convert the entries of the codebooks to integers
    cb1, cb2 = integer_list(cb1), integer_list(cb2)
    
    #generate all adversary words of length n
    adv = generate_words(n, adv=True)
    
    #possible triples of codeword 1, codeword 2, and adversary word
    triples = list(product(cb1, cb2, adv))
    
    #find the possible (non) adversarial sums
    initial_sums = [sum(Q[:2]) for Q in triples]
    adversarial_sums = [sum(Q) for Q in triples]
    coords = [Q[:2] for Q in triples]
    
    #create sets from the sums lists so that we can take intersections
    initial_sums_set = set(initial_sums)
    print(f'non-adversarial sums = {zero_pad_iter(n, initial_sums_set)}')
    adversarial_sums_set = set(adversarial_sums)
    
    #check if there is a sum pair appearing twice the no-adversary set
    check_non_adv_repeat = (len(initial_sums_set) != len(cb1)*len(cb2))
    #check if non-adversarial and adversarial sums have intersection
    check_adv_non_adv_intersect = (len(initial_sums_set.intersection(adversarial_sums_set)) != 0)
    
    #if either bad case occurs, print the issue, then return False
    if check_non_adv_repeat or check_adv_non_adv_intersect:
        if check_non_adv_repeat:
            print(f'False: non-adversarial sums are not unique. |{zero_pad_iter(n, initial_sums_set)}| != {len(cb1)*len(cb2)}')
        if check_adv_non_adv_intersect:
            print(f'False: non-adversarial intersection with adversarial is {zero_pad_iter(n, initial_sums_set.intersection(adversarial_sums_set))}')
        return False
    
    #otherwise, we check whether one codeword can always be recovered in the adversarial setting
    else:
        
        unique_adversarial_sums = unique(adversarial_sums)

        # Check that repeated outputs all match on at least one of the two users
        for index in range(len(unique_adversarial_sums)):
            
            #for each unique adversarial sum, collect all user 1 and user 2 codewords that contribute to create that sum
            adv_sum = unique_adversarial_sums[index]
            user_1 = [coords[k][0] for k in range(len(adversarial_sums)) if adversarial_sums[k] == adv_sum]
            user_2 = [coords[k][1] for k in range(len(adversarial_sums)) if adversarial_sums[k] == adv_sum]
            
            unique_user_1 = unique(user_1)
            unique_user_2 = unique(user_2)
                                                  
            # if both users have multiple codewords that could result in that sum, we can't decode either user
            if (len(unique_user_1) > 1) and (len(unique_user_2) > 1):
                print(f'False: possible codewords from codebook 1 resulting in {zero_pad(n, adv_sum)} are {zero_pad_iter(n, unique_user_1)}')
                print(f'possible codewords from codebook 2 resulting in {zero_pad(n, adv_sum)} are {zero_pad_iter(n, unique_user_2)}')
                return False
        return True                                            

Example calls
---

Below is an example call with a good codebook pair of block length 5.

In [4]:
good_pair_check(['10111','01100'],['01111','10000'],5)

non-adversarial sums = ['02211', '11100', '11222', '20111']


True

Next, some calls with bad pairs.

In [5]:
#a pair that doesn't have unique non-adversarial sums
good_pair_check(['10111','01100', '01100'],['01111','10000'],5)

non-adversarial sums = ['02211', '11100', '11222', '20111']
False: non-adversarial sums are not unique. |['02211', '11100', '11222', '20111']| != 6


False

In [6]:
#a pair where the non-adversarial with adversarial sums have nonempty intersection
good_pair_check(['10111','01100', '00011'],['01111','10000'],5)

non-adversarial sums = ['01122', '02211', '20111', '11222', '10011', '11100']
False: non-adversarial intersection with adversarial is ['11222', '20111']


False

In [7]:
#a pair that doesn't have unique non-adversarial sums AND has nonempty adv/non-adv intersection
good_pair_check(['10111','01100', '01100', '00011'],['01111','10000'],5)

non-adversarial sums = ['01122', '02211', '20111', '11222', '10011', '11100']
False: non-adversarial sums are not unique. |['01122', '02211', '20111', '11222', '10011', '11100']| != 8
False: non-adversarial intersection with adversarial is ['11222', '20111']


False

In [8]:
#a pair that passes the first two but not the one user recoverable condition
good_pair_check(['10010','01100','00001'],['00110','11000','00001'],5)

non-adversarial sums = ['00002', '12100', '10120', '01101', '00111', '21010', '11001', '01210', '10011']
False: possible codewords from codebook 1 resulting in 21011 are ['00001', '10010']
possible codewords from codebook 2 resulting in 21011 are ['11000', '00001']


False

More Than Two Users
---

Now, we need a parameter in the function for the fraction of users we want to correct.

**The following is in progress, and needs to be tested a bit more before I'm sure it's working**

In [ ]:
#here's a revision for (possibly) more users. 
#if I do it right, this should also work for two users
def good_codebooks_check(codebooks, n, gamma):
    #codebooks is a list of lists of binary words of binary length n
    #binary codewords in list can be input as strings to allow leading zeros
    #gamma is the fractional threshold to correct. It should be a float between 0 and 1, inclusive.
    
    #convert the entries of the codebooks to integers
    int_codebooks = [integer_list(cb) for cb in codebooks]
    num_codebooks = len(int_codebooks)
    
    #generate all adversary words of length n
    adv = generate_words(n, adv=True)
    
    int_codebooks.append(adv)
    
    #possible combinations of codeword 1, codeword 2, ..., codeword u, and adversary word
    transmitted = list(product(*int_codebooks))
    
    #find the possible (non) adversarial sums
    initial_sums = [sum(Q[:num_codebooks]) for Q in transmitted]
    adversarial_sums = [sum(Q) for Q in transmitted]
    #for each sum, record the codewords contributing to that sum
    coords = [Q[:num_codebooks] for Q in transmitted]
    
    #create sets from the sums lists so that we can take intersections
    initial_sums_set = set(initial_sums)
    print(f'non-adversarial sums = {zero_pad_iter(n, initial_sums_set)}')
    adversarial_sums_set = set(adversarial_sums)
    
    #check if there is a sum pair appearing twice the no-adversary set
    correct_num_no_adv = math.prod([len(cb) for cb in codebooks])
    check_non_adv_repeat = (len(initial_sums_set) != correct_num_no_adv)
    
    #check if non-adversarial and adversarial sums have intersection
    check_adv_non_adv_intersect = (len(initial_sums_set.intersection(adversarial_sums_set)) != 0)
    
    #if either bad case occurs, print the issue, then return False
    if check_non_adv_repeat or check_adv_non_adv_intersect:
        if check_non_adv_repeat:
            print(f'False: non-adversarial sums are not unique. |{zero_pad_iter(n, initial_sums_set)}| != {correct_num_no_adv}')
        if check_adv_non_adv_intersect:
            print(f'False: non-adversarial intersection with adversarial is {zero_pad_iter(n, initial_sums_set.intersection(adversarial_sums_set))}')
        return False
    
    #otherwise, we check whether gamma*u codewords can always be recovered in the adversarial setting
    else:
        #need to cover at least gamma times the number of codebooks, so round up to the next int
        to_recover = math.ceil(gamma * num_codebooks)
        
        unique_adversarial_sums = unique(adversarial_sums)

        # Check that repeated outputs all match on at least one of the two users
        for index in range(len(unique_adversarial_sums)):
            
            #for each unique adversarial sum, collect all user codewords that contribute to create that sum
            adv_sum = unique_adversarial_sums[index]
            user_words = list(np.zeros(num_codebooks))
            for ii in range(num_codebooks):
                user_words[ii] = [coords[k][ii] for k in range(len(adversarial_sums)) if adversarial_sums[k] == adv_sum]
            
            unique_user_words = [unique(ii) for ii in user_words]
           
            num_unique_words = [len(jj) for jj in unique_user_words]
           
            num_recoverable = np.count_nonzero(np.array(num_unique_words) == 1)
            print(num_recoverable)
            if num_recoverable < to_recover:
                # print(f'False: possible codewords from codebook 1 resulting in {zero_pad(n, adv_sum)} are {zero_pad_iter(n, unique_user_1)}')
                # print(f'possible codewords from codebook 2 resulting in {zero_pad(n, adv_sum)} are {zero_pad_iter(n, unique_user_2)}')
                return False
        return True     

In [ ]:
#Here is where I will add some examples